In [50]:
import cv2
import mediapipe as mp
import numpy as np
import os

In [51]:
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_drawing = mp.solutions.drawing_utils

I0000 00:00:1736513125.441712 17077204 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 83), renderer: Apple M2


In [52]:
def calculate_angle(a, b, c):
    
    a = np.array(a)  # First point
    b = np.array(b)  # Midpoint (vertex)
    c = np.array(c)  # End point

    # Vectors formed by the points
    ba = a - b  # Vector from b to a
    bc = c - b  # Vector from b to c

    # Calculate cosine of the angle using the dot product and magnitudes
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))

    # Clip to handle numerical errors (cosine should be in range [-1, 1])
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)

    # Calculate the angle in radians and convert to degrees
    angle = np.arccos(cosine_angle) * (180.0 / np.pi)

    return angle


W0000 00:00:1736513125.564752 18775655 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [53]:
calculate_angle((-1,-1),(0,0),(1,0))


W0000 00:00:1736513125.583067 18775661 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


135.0

In [54]:
def annotate_angle(image, a, b, c, angle_text, color=(255, 0, 0)):
    angle = calculate_angle(a, b, c)
    b_coords = (int(b[0] * image.shape[1]), int(b[1] * image.shape[0]))  # Midpoint coordinates
    cv2.putText(image, f"{angle_text}: {int(angle)}", b_coords, cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    return angle

In [55]:
def analyze_squaring_up(image, landmarks, side="right"):
    feedback = []

    # Dynamically choose landmarks
    shoulder = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_SHOULDER").value]
    elbow = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_ELBOW").value]
    wrist = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_WRIST").value]
    hip = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_HIP").value]
    knee = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_KNEE").value]
    ankle = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_ANKLE").value]

    # Draw pose landmarks
    mp_drawing.draw_landmarks(image, result.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                              mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2))

    # Elbow angle
    elbow_angle = annotate_angle(
        image, [shoulder.x, shoulder.y], [elbow.x, elbow.y], [wrist.x, wrist.y],
        "Elbow", color=(0, 255, 0) if 85 <= calculate_angle([shoulder.x, shoulder.y], [elbow.x, elbow.y], [wrist.x, wrist.y]) <= 95 else (0, 0, 255)
    )

    if elbow_angle < 85 or elbow_angle > 95:
        feedback.append(f"The player's elbow should be closer to a 90-degree angle.")

    # Knee angle
    knee_angle = annotate_angle(
        image, [hip.x, hip.y], [knee.x, knee.y], [ankle.x, ankle.y],
        "Knee", color=(0, 255, 0) if calculate_angle([hip.x, hip.y], [knee.x, knee.y], [ankle.x, ankle.y]) <= 160 else (0, 0, 255)
    )
    if knee_angle > 160:
        feedback.append(f"The player's should bend their knees more.")

    # # Back alignment
    # back_angle = annotate_angle(
    #     image, [shoulder.x, shoulder.y], [hip.x, hip.y], [knee.x, knee.y],
    #     "Back", color=(0, 255, 0) if calculate_angle([shoulder.x, shoulder.y], [hip.x, hip.y], [knee.x, knee.y]) >= 170 else (0, 0, 255)
    # )
    # if back_angle < 170:
    #     feedback.append(f"The player's back should be straighter.")

    return feedback, image

In [56]:
def analyze_release(image, landmarks, side="right"):
    feedback = []

    # Dynamically choose landmarks
    shoulder = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_SHOULDER").value]
    elbow = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_ELBOW").value]
    wrist = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_WRIST").value]
    hip = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_HIP").value]
    knee = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_KNEE").value]
    ankle = landmarks[getattr(mp_pose.PoseLandmark, f"{side.upper()}_ANKLE").value]

    # Draw pose landmarks
    mp_drawing.draw_landmarks(image, result.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
                              mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=2))

    # Arm straightness
    arm_angle = annotate_angle(
        image, [shoulder.x, shoulder.y], [elbow.x, elbow.y], [wrist.x, wrist.y],
        "Arm", color=(0, 255, 0) if calculate_angle([shoulder.x, shoulder.y], [elbow.x, elbow.y], [wrist.x, wrist.y]) <= 10 else (0, 0, 255)
    )
    if arm_angle > 10:
        feedback.append(f"The player's arm should be straighter during the release.")

    # Leg straightness
    leg_angle = annotate_angle(
        image, [hip.x, hip.y], [knee.x, knee.y], [ankle.x, ankle.y],
        "Leg", color=(0, 255, 0) if calculate_angle([hip.x, hip.y], [knee.x, knee.y], [ankle.x, ankle.y]) >= 170 else (0, 0, 255)
    )
    if leg_angle < 170:
        feedback.append(f"The player's leg should be straighter during the release.")

    # # Back alignment
    # back_angle = annotate_angle(
    #     image, [shoulder.x, shoulder.y], [hip.x, hip.y], [knee.x, knee.y],
    #     "Back", color=(0, 255, 0) if calculate_angle([shoulder.x, shoulder.y], [hip.x, hip.y], [knee.x, knee.y]) >= 170 else (0, 0, 255)
    # )
    # if back_angle < 170:
    #     feedback.append(f"The player's back should be straighter.")

    return feedback, image

In [57]:
squared_up_path = "results/squared_up_frame.jpg"
release_path = "results/release_frame.jpg"

In [58]:
side = "right"

In [66]:
if os.path.exists(squared_up_path):
    squared_up_image = cv2.imread(squared_up_path)
    result = pose.process(cv2.cvtColor(squared_up_image, cv2.COLOR_BGR2RGB))

    if result.pose_landmarks:
        feedback, analyzed_image = analyze_squaring_up(squared_up_image, result.pose_landmarks.landmark, side)
        cv2.imwrite("results/squared_up_feedback.jpg", analyzed_image)
        print("Squaring Up Feedback:")
        for item in feedback:
            print(item)

Squaring Up Feedback:
The player's elbow should be closer to a 90-degree angle.


In [67]:
if os.path.exists(release_path):
    release_image = cv2.imread(release_path)
    result = pose.process(cv2.cvtColor(release_image, cv2.COLOR_BGR2RGB))

    if result.pose_landmarks:
        feedback, analyzed_image = analyze_release(release_image, result.pose_landmarks.landmark, side)
        cv2.imwrite("results/release_feedback.jpg", analyzed_image)
        print("\nRelease Feedback:")
        for item in feedback:
            print(item)



Release Feedback:
The player's arm should be straighter during the release.
The player's leg should be straighter during the release.


SyntaxError: invalid syntax (2830201818.py, line 1)